In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from helpers.data_loaders import *
from torch_geometric.nn import Node2Vec
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os 


from IPython.display import display

sns.set_style('whitegrid')


In [2]:
movies_df, ratings_df = load_movielens_data()

print('Movies Dataset:')
display(movies_df.head())

print('Ratings Dataset:')
display(ratings_df.head())


Movies Dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings Dataset:


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [3]:
movielens_graph = transform_movielenses_to_graph(ratings_df.copy(), movies_df.copy())

print(f"Number of user nodes: {len(movielens_graph['user_nodes'])}")
print(f"Number of item nodes: {len(movielens_graph['item_nodes'])}")
print(f"Number of genre nodes: {len(movielens_graph['genre_nodes'])}")
print(f"Number of user-item edges: {len(movielens_graph['user_item_edges'])}")
print(f"Number of item-genre edges: {len(movielens_graph['item_genre_edges'])}")


Number of user nodes: 162342
Number of item nodes: 62423
Number of genre nodes: 20
Number of user-item edges: 12452811
Number of item-genre edges: 112307


In [4]:
reviews_df_steam_cleaned, items_df_steam_cleaned = load_steam_data()

print("Cleaned Reviews Dataset:")
display(reviews_df_steam_cleaned.head())
print("Cleaned Items Dataset:")
display(items_df_steam_cleaned.head())


Cleaned Reviews Dataset:


,user_id,app_id
0,76561197970982479,1250
1,76561197970982479,22200
4,js41637,227300
5,js41637,239030
6,evcentric,248820


Cleaned Items Dataset:


,app_id,title,genres
0,761140,Lost Summoner Kitty,"[Action, Casual, Indie, Simulation, Strategy]"
1,643980,Ironbound,"[Free to Play, Indie, RPG, Strategy]"
2,670290,Real Pool 3D - Poolians,"[Casual, Free to Play, Indie, Simulation, Sports]"
3,767400,弹炸人2222,"[Action, Adventure, Casual]"
5,772540,Battle Royale Trainer,"[Action, Adventure, Simulation]"


In [5]:
steam_graph = transform_steam_to_graph(reviews_df_steam_cleaned, items_df_steam_cleaned)

print(f"Number of user nodes: {len(steam_graph['user_nodes'])}")
print(f"Number of item nodes: {len(steam_graph['item_nodes'])}")
print(f"Number of genre nodes: {len(steam_graph['genre_nodes'])}")
print(f"Number of user-item edges: {len(steam_graph['user_item_edges'])}")
print(f"Number of item-genre edges: {len(steam_graph['item_genre_edges'])}")


Number of user nodes: 22077
Number of item nodes: 2802
Number of genre nodes: 23
Number of user-item edges: 45261
Number of item-genre edges: 72786


In [6]:
merged_graph = merge_graphs(movielens_graph, steam_graph)

print(f"Number of user nodes in merged graph: {len(merged_graph['user_nodes'])}")
print(f"Number of item nodes in merged graph: {len(merged_graph['item_nodes'])}")
print(f"Number of genre nodes in merged graph: {len(merged_graph['genre_nodes'])}")
print(f"Number of user-item edges in merged graph: {len(merged_graph['user_item_edges'])}")
print(f"Number of item-genre edges in merged graph: {len(merged_graph['item_genre_edges'])}")


Number of user nodes in merged graph: 184419
Number of item nodes in merged graph: 65225
Number of genre nodes in merged graph: 41
Number of user-item edges in merged graph: 12497401
Number of item-genre edges in merged graph: 185092


In [7]:
G_pyg, node_to_int_id, int_id_to_node = create_pyg_graph(merged_graph)


Data(edge_index=[2, 25233232], num_nodes=249685)


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
model_path = "models/best_model_79_20251115_175000.pth"

if os.path.exists(model_path):
    print(f"Loading pre-trained model from {model_path}...")
    model = Node2Vec(
        edge_index=G_pyg.edge_index,
        embedding_dim=64,
        walk_length=20,
        context_size=10,
        walks_per_node=20,
        num_negative_samples=1,
        p=1,
        q=0.5,
        sparse=True,
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    embeddings = model.embedding.weight
    print("Model loaded and embeddings are ready.")
else:
    print("No pre-trained model found. Initializing and training a new model...")
    model = Node2Vec(
        edge_index=G_pyg.edge_index,
        embedding_dim=64,
        walk_length=20,
        context_size=10,
        walks_per_node=20,
        num_negative_samples=1,
        p=1,
        q=0.5,
        sparse=True,
    ).to(device)

    print("Model initialization and walk generation complete.")

    print("Creating data loader...")
    loader = model.loader(batch_size=128, shuffle=True, num_workers=4)
    optimizer = torch.optim.SparseAdam(model.parameters(), lr=0.01)

    lr_scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.1,
        patience=3, 
    )

    early_stopping_patience = 5
    epochs_no_improve = 0
    best_loss = float('inf')  
    max_epochs = 101 

    print("Starting training...")
    for epoch in range(1, max_epochs):
        model.train()
        total_loss = 0
        for pos_rw, neg_rw in loader:
            optimizer.zero_grad()
            loss = model.loss(pos_rw.to(device), neg_rw.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch: {epoch}, Loss: {avg_loss:.4f}, LR: {current_lr}')

        lr_scheduler.step(avg_loss)

        if avg_loss < best_loss:
            best_loss = avg_loss
            epochs_no_improve = 0
            date = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
            torch.save(model.state_dict(), f'models/best_model_{epoch}_{date}.pth')
        else:
            epochs_no_improve += 1

        if epochs_no_improve == early_stopping_patience:
            print(f"Early stopping triggered after {epoch} epochs.")
            break 

    model.eval()
    embeddings = model.embedding.weight
    print("Training complete. Embeddings are ready.")


Loading pre-trained model from models/best_model_79_20251115_175000.pth...
Model loaded and embeddings are ready.


In [10]:
def get_recommendation_score(user_id, item_id, embeddings, mapping):
    try:
        u_id = mapping[user_id]
        v_id = mapping[item_id]

        u_emb = embeddings[u_id].cpu().detach()
        v_emb = embeddings[v_id].cpu().detach()

        return torch.dot(u_emb, v_emb).item()

    except KeyError:
        return "Node not in mapping"
    except Exception as e:
        return f"Error: {e}"

real_edges_to_test = [
    merged_graph['user_item_edges'][i]
    for i in np.random.choice(len(merged_graph['user_item_edges']), 5)
]

print("--- Testing REAL Edges ---")
for user, item in real_edges_to_test:
    score = get_recommendation_score(user, item, embeddings, node_to_int_id)
    print(f"Score for ({user}, {item}): {score:.4f}")

user_nodes = merged_graph['user_nodes']
item_nodes = merged_graph['item_nodes']
all_real_edges_set = set(merged_graph['user_item_edges'])

fake_edges_to_test = []
while len(fake_edges_to_test) < 5:
    rand_user = np.random.choice(user_nodes)
    rand_item = np.random.choice(item_nodes)

    if (rand_user, rand_item) not in all_real_edges_set:
        fake_edges_to_test.append((rand_user, rand_item))

print("\n--- Testing FAKE Edges ---")
for user, item in fake_edges_to_test:
    score = get_recommendation_score(user, item, embeddings, node_to_int_id)
    print(f"Score for ({user}, {item}): {score:.4f}")


--- Testing REAL Edges ---
Score for (MovieLens_user_25349, MovieLens_item_64620): 3.7422
Score for (MovieLens_user_62125, MovieLens_item_3784): 4.8717
Score for (MovieLens_user_117863, MovieLens_item_4993): 6.2082
Score for (MovieLens_user_121010, MovieLens_item_3751): 4.2332
Score for (MovieLens_user_5591, MovieLens_item_1073): 4.8986

--- Testing FAKE Edges ---
Score for (MovieLens_user_12319, MovieLens_item_124566): -0.2744
Score for (MovieLens_user_131030, MovieLens_item_207039): -0.0590
Score for (MovieLens_user_120769, MovieLens_item_123310): -0.2544
Score for (MovieLens_user_151550, MovieLens_item_132064): 0.0864
Score for (MovieLens_user_134939, MovieLens_item_146580): -0.0535


In [11]:
!uv pip freeze


Using Python 3.10.19 environment at: /home/berni/education/Generic-Recommender-Semantics-Put/.venv
aiohappyeyeballs==2.6.1
aiohttp==3.13.2
aiosignal==1.4.0
asttokens==3.0.0
async-timeout==5.0.1
attrs==25.4.0
certifi==2025.11.12
charset-normalizer==3.4.4
comm==0.2.3
contourpy==1.3.2
cycler==0.12.1
debugpy==1.8.17
decorator==5.2.1
exceptiongroup==1.3.0
executing==2.2.1
filelock==3.20.0
fonttools==4.60.1
frozenlist==1.8.0
fsspec==2025.10.0
idna==3.11
ipykernel==7.1.0
ipython==8.37.0
ipywidgets==8.1.8
jedi==0.19.2
jinja2==3.1.6
jupyter-client==8.6.3
jupyter-core==5.9.1
jupyterlab-widgets==3.0.16
kiwisolver==1.4.9
markupsafe==3.0.3
matplotlib==3.10.7
matplotlib-inline==0.2.1
mpmath==1.3.0
multidict==6.7.0
nest-asyncio==1.6.0
networkx==3.4.2
numpy==2.2.6
nvidia-cublas-cu12==12.4.5.8
nvidia-cuda-cupti-cu12==12.4.127
nvidia-cuda-nvrtc-cu12==12.4.127
nvidia-cuda-runtime-cu12==12.4.127
nvidia-cudnn-cu12==9.1.0.70
nvidia-cufft-cu12==11.2.1.3
nvidia-cufile-cu12==1.13.1.3
nvidia-curand-cu12==10.3.5